In [0]:
# ============================================================
# CELL 1 — IMPORTS + CONFIG
# ============================================================

import os
import datetime as dt
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/btc_usd/silver/mlflow_vol"

TICKER       = "BTC-USD"
CATALOG      = "btc_usd"
SILVER_TABLE = f"{CATALOG}.silver.cleaned"
GOLD_TABLE   = f"{CATALOG}.gold.ml_analytics"
EXPERIMENT   = f"/Shared/{CATALOG}_ml"

FEATURE_COLS = [
    "return_1h", "lag_return_2h", "lag_return_3h",
    "lag_return_6h", "lag_return_24h",
    "rsi14", "ma7", "ma21",
    "bb_width", "bb_upper", "bb_lower",
    "volatility_24h", "momentum_6h",
    "volume_ratio", "hour_of_day", "day_of_week", "volume"
]

LABEL_COL = "label_return_1h_ahead"

In [0]:
# ============================================================
# CELL 2 — FUNCTION: prepare_features()
# Reads Silver, adds target label, splits train/test
# ============================================================

def prepare_features(spark):
    """
    Reads Silver table, adds prediction target, splits
    into train/test using time-based split (80/20).

    Returns:
        train_df, test_df : Spark DataFrames
    """
    print("📖 Reading Silver table...")
    df = spark.table(SILVER_TABLE).orderBy("datetime")

    # Add target: next hour's return
    w_order = Window.orderBy("datetime")
    df = df.withColumn(
        LABEL_COL,
        F.lead("return_1h", 1).over(w_order)
    )

    # Drop rows where label or features are null
    df = df.dropna(subset=FEATURE_COLS + [LABEL_COL])

    # Time-based train/test split — 80/20
    n       = df.count()
    train_n = int(n * 0.8)

    df = df.withColumn("row_num", F.row_number().over(w_order))
    train_df = df.filter(F.col("row_num") <= train_n).drop("row_num")
    test_df  = df.filter(F.col("row_num") >  train_n).drop("row_num")

    print(f"✅ Total rows  : {n}")
    print(f"   Train rows  : {train_df.count()} | {train_df.agg(F.min('datetime'), F.max('datetime')).first()}")
    print(f"   Test rows   : {test_df.count()}  | {test_df.agg(F.min('datetime'), F.max('datetime')).first()}")

    return train_df, test_df

In [0]:
# ============================================================
# CELL 3 — FUNCTION: train_model()
# Trains LR, Random Forest, GBT — logs all to MLflow
# Returns best model based on direction accuracy
# ============================================================

from pyspark.ml.regression import (
    LinearRegression,
    RandomForestRegressor,
    GBTRegressor
)

def _evaluate(preds, label_col):
    """Helper — computes all metrics for a set of predictions."""
    rmse = RegressionEvaluator(
        labelCol=label_col, predictionCol="prediction", metricName="rmse"
    ).evaluate(preds)

    r2 = RegressionEvaluator(
        labelCol=label_col, predictionCol="prediction", metricName="r2"
    ).evaluate(preds)

    mae = RegressionEvaluator(
        labelCol=label_col, predictionCol="prediction", metricName="mae"
    ).evaluate(preds)

    direction_acc = (
        preds
        .select(
            F.when(F.col("prediction") >= 0, 1).otherwise(0).alias("pred_up"),
            F.when(F.col(label_col)    >= 0, 1).otherwise(0).alias("label_up")
        )
        .withColumn("correct", (F.col("pred_up") == F.col("label_up")).cast("double"))
        .agg(F.avg("correct"))
        .first()[0]
    )

    return {
        "rmse":               float(rmse),
        "r2":                 float(r2),
        "mae":                float(mae),
        "direction_accuracy": float(direction_acc)
    }


def _train_and_log(name, regressor, assembler, scaler, train_df, test_df, spark):
    """
    Helper — builds pipeline, trains, evaluates, logs to MLflow.
    Returns (model, preds, run_id, metrics)
    """
    pipeline = Pipeline(stages=[assembler, scaler, regressor])

    mlflow.set_experiment(EXPERIMENT)

    with mlflow.start_run(
        run_name=f"btc_{name}_{dt.datetime.now().strftime('%Y%m%d_%H%M')}"
    ) as run:

        # Log params
        mlflow.log_param("ticker",   TICKER)
        mlflow.log_param("model",    name)
        mlflow.log_param("features", ",".join(FEATURE_COLS))
        mlflow.log_param("label",    LABEL_COL)
        mlflow.log_param("train_rows", train_df.count())
        mlflow.log_param("test_rows",  test_df.count())

        # Log model-specific params
        if hasattr(regressor, 'getRegParam'):
            mlflow.log_param("regParam",        regressor.getRegParam())
            mlflow.log_param("elasticNetParam",  regressor.getElasticNetParam())
        if hasattr(regressor, 'getNumTrees'):
            mlflow.log_param("numTrees",         regressor.getNumTrees())
            mlflow.log_param("maxDepth",         regressor.getMaxDepth())
        if hasattr(regressor, 'getMaxIter') and not hasattr(regressor, 'getRegParam'):
            mlflow.log_param("maxIter",          regressor.getMaxIter())
            mlflow.log_param("maxDepth",         regressor.getMaxDepth())

        # Train
        print(f"   Training {name}...")
        model  = pipeline.fit(train_df)
        preds  = model.transform(test_df)

        # Evaluate
        metrics = _evaluate(preds, LABEL_COL)
        for k, v in metrics.items():
            mlflow.log_metric(k, v)

        # Log model
        signature = infer_signature(
            train_df.select(FEATURE_COLS).toPandas(),
            preds.select("prediction").toPandas()
        )
        mlflow.spark.log_model(
            model,
            artifact_path="model",
            signature=signature
        )

        run_id = run.info.run_id

        print(f"   ✅ {name} complete!")
        print(f"      Run ID             : {run_id}")
        print(f"      RMSE               : {metrics['rmse']:.6f}")
        print(f"      R²                 : {metrics['r2']:.4f}")
        print(f"      MAE                : {metrics['mae']:.6f}")
        print(f"      Direction Accuracy : {metrics['direction_accuracy']:.2%}")

    return model, preds, run_id, metrics


def train_model(train_df, test_df, spark):
    """
    Trains Linear Regression, Random Forest, and GBT.
    Logs all 3 runs to MLflow.
    Returns the best model based on direction accuracy.
    """
    # Shared pipeline stages
    assembler = VectorAssembler(
        inputCols=FEATURE_COLS,
        outputCol="features_raw"
    )
    scaler = StandardScaler(
        inputCol="features_raw",
        outputCol="features",
        withMean=True,
        withStd=True
    )

    # Define all 3 models
    models = {
            "linear_regression": LinearRegression(
                featuresCol="features",
                labelCol=LABEL_COL,
                regParam=0.01,
                elasticNetParam=0.0,
                maxIter=100
            ),
            "random_forest": RandomForestRegressor(
                featuresCol="features",
                labelCol=LABEL_COL,
                numTrees=200,        # more trees
                maxDepth=4,          # shallower = less overfit
                minInstancesPerNode=10,  # prevent tiny splits
                featureSubsetStrategy="sqrt",  # use sqrt features per split
                seed=42
            ),
            "gbt_tuned": GBTRegressor(
                featuresCol="features",
                labelCol=LABEL_COL,
                maxIter=200,         # more trees
                maxDepth=3,          # much shallower — was 5, now 3
                stepSize=0.05,       # slower learning rate — was 0.1
                minInstancesPerNode=20,  # prevent overfitting on small splits
                subsamplingRate=0.8, # use 80% of data per tree (like dropout)
                seed=42
            )
        }

    # Train all 3
    print("\n🤖 Training 3 models...")
    print("=" * 55)

    results = {}
    for name, regressor in models.items():
        print(f"\n🔧 [{name}]")
        model, preds, run_id, metrics = _train_and_log(
            name, regressor, assembler, scaler, train_df, test_df, spark
        )
        results[name] = {
            "model":   model,
            "preds":   preds,
            "run_id":  run_id,
            "metrics": metrics
        }

    # Compare all 3
    print("\n" + "=" * 55)
    print("📊 Model Comparison:")
    print(f"{'Model':<25} {'RMSE':>10} {'R²':>8} {'MAE':>10} {'Dir Acc':>10}")
    print("-" * 55)
    for name, result in results.items():
        m = result["metrics"]
        print(
            f"{name:<25} "
            f"{m['rmse']:>10.6f} "
            f"{m['r2']:>8.4f} "
            f"{m['mae']:>10.6f} "
            f"{m['direction_accuracy']:>10.2%}"
        )

    # Pick best model by direction accuracy
    best_name = max(results, key=lambda x: results[x]["metrics"]["direction_accuracy"])
    best      = results[best_name]

    print(f"\n🏆 Best model: {best_name.upper()} "
          f"(Direction Accuracy: {best['metrics']['direction_accuracy']:.2%})")

    return best["model"], best["preds"], best["run_id"], best_name

In [0]:
# ============================================================
# CELL 4 — FUNCTION: store_gold()
# Saves predictions + metrics to gold.ml_analytics
# ============================================================

def store_gold(preds, run_id, spark):
    """
    Saves model predictions and metadata to Gold table.
    This is what the dashboard will read.

    Columns saved:
        datetime         : timestamp of the candle
        close            : actual close price
        label            : actual next hour return
        prediction       : model's predicted return
        pred_direction   : 1 = predicted up, 0 = predicted down
        actual_direction : 1 = actually went up, 0 = went down
        correct          : 1 = direction correct, 0 = wrong
        run_id           : MLflow run ID for traceability
        trained_at       : when this model was trained
    """
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

    gold = (
        preds
        .select(
            "datetime",
            "close",
            F.col(LABEL_COL).alias("label"),
            "prediction"
        )
        .withColumn("pred_direction",
            F.when(F.col("prediction") >= 0, 1).otherwise(0)
        )
        .withColumn("actual_direction",
            F.when(F.col("label") >= 0, 1).otherwise(0)
        )
        .withColumn("correct",
            (F.col("pred_direction") == F.col("actual_direction")).cast("integer")
        )
        .withColumn("run_id",     F.lit(run_id))
        .withColumn("trained_at", F.lit(dt.datetime.now(dt.timezone.utc).isoformat()))
    )

    gold.write.mode("overwrite").saveAsTable(GOLD_TABLE)
    count = gold.count()
    print(f"🥇 Gold [{GOLD_TABLE}]: wrote {count} rows.")

    # Gold summary — this is what the dashboard will show
    print(f"\n📊 Gold Summary:")
    spark.table(GOLD_TABLE).select(
        F.count("*").alias("total_predictions"),
        F.round(F.avg("correct"),      4).alias("direction_accuracy"),
        F.round(F.avg("prediction"),   6).alias("avg_predicted_return"),
        F.round(F.avg("label"),        6).alias("avg_actual_return"),
        F.round(F.stddev("prediction"),6).alias("prediction_stddev"),
        F.min("datetime").alias("from"),
        F.max("datetime").alias("to"),
    ).display()

    # Sample predictions
    print(f"\n🔍 Sample Predictions (latest 10):")
    spark.table(GOLD_TABLE) \
        .orderBy(F.desc("datetime")) \
        .limit(10) \
        .select(
            "datetime", "close",
            F.round("label",      6).alias("actual_return"),
            F.round("prediction", 6).alias("predicted_return"),
            "pred_direction", "actual_direction", "correct"
        ).display()

In [0]:
def run_ml_job(spark):
    print("=" * 55)
    print(f"ML Job | {TICKER} | {dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print("=" * 55)

    print("\n📐 Preparing features...")
    train_df, test_df = prepare_features(spark)

    print("\n🤖 Training models...")
    model, preds, run_id, best_name = train_model(train_df, test_df, spark)

    print(f"\n💾 Saving best model ({best_name}) predictions to Gold...")
    store_gold(preds, run_id, spark)

    print(f"\n🏁 ML Job complete! Best model: {best_name} | Run ID: {run_id}")
    return run_id

In [0]:
run_id = run_ml_job(spark)

In [0]:
# # Clear all cached models
# import gc

# try:
#     del model
# except: pass

# try:
#     del loaded_model
# except: pass

# try:
#     del pipeline_model
# except: pass

# # Force garbage collection
# gc.collect()

# print("✅ Model cache cleared.")

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS btc_usd;

CREATE SCHEMA IF NOT EXISTS btc_usd.silver;

CREATE VOLUME IF NOT EXISTS btc_usd.silver.mlflow_vol;

In [0]:
spark.catalog.clearCache()